In [7]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import math

In [2]:
# Import data from the base notebook
import import_ipynb
from IPython.utils.capture import capture_output
import xraymodel

c:\Users\cschmid\AppData\Local\Programs\Python\Python313\Lib\site-packages\nbformat\__init__.py:96: MissingIDFieldWarning: Cell is missing an id field, this will become a hard error in future nbformat versions. You may want to use `normalize()` on your notebooks before validations (available since nbformat 5.1.4). Previous versions of nbformat are fixing this issue transparently, and will stop doing so in the future.
  validate(nb)


found 64534 images
{np.str_('Pleural Other'), np.str_('Support Devices')} will be dropped
Loaded 10000 images


NameError: name 'make_subplots' is not defined

In [3]:
# Use "capture" do suppress plotting images
with capture_output() as captured:
    %run r1_false_positive_subgroup_investigation.ipynb

In [14]:

PATHOLOGY = "Atelectasis"
FP_THRESHOLD = 0.7
RANDOM_SEED = 42


# ------------------------------------------------------------
# 1. Build image-level dataframe
# ------------------------------------------------------------

label_idx = DATASET_LABEL_INDICES_MAP[PATHOLOGY]

truth = np.asarray(
    XRAY_LABELS_LIST[:, label_idx],
    dtype=float
)

score = np.asarray(
    MODEL_SCORE_PREDICTIONS_CONVERTED[:, label_idx],
    dtype=float
)

valid = (
    np.isfinite(truth)
    & np.isfinite(score)
    & np.isin(truth, [0, 1])
)

image_df = meta.copy().reset_index(drop=True)

image_df["y_true"] = truth
image_df["score"] = score
image_df["y_pred"] = (score >= FP_THRESHOLD).astype(int)

image_df["Outcome"] = np.select(
    [
        (truth == 0) & (score >= FP_THRESHOLD),
        (truth == 0) & (score < FP_THRESHOLD),
        (truth == 1) & (score >= FP_THRESHOLD),
        (truth == 1) & (score < FP_THRESHOLD),
    ],
    [
        "FP",
        "TN",
        "TP",
        "FN",
    ],
    default="Invalid"
)

image_df["valid"] = valid


# Keep only binary-label cases and AP/PA
d = image_df[
    image_df["valid"]
    & image_df["View"].isin(["AP", "PA"])
    & image_df["Outcome"].isin(["FP", "TN", "TP"])
].copy()

print("Number of valid AP/PA negative cases:", len(d))
print(d.groupby(["View", "Outcome"]).size())


# ------------------------------------------------------------
# 2. Plot FP rate: AP vs PA — Plotly
# ------------------------------------------------------------

fp_rates = (
    d.assign(is_fp=d["Outcome"].eq("FP"))
     .groupby("View")
     .agg(
         N=("is_fp", "size"),
         FP=("is_fp", "sum")
     )
     .reset_index()
)

fp_rates["FP_rate"] = fp_rates["FP"] / fp_rates["N"]


fig = px.bar(
    fp_rates,
    x="View",
    y="FP_rate",
    text="FP_rate",
    hover_data={
        "N": True,
        "FP": True,
        "FP_rate": ":.2%"
    },
    title=f"{PATHOLOGY} — False-positive rate by view",
    labels={
        "View": "View",
        "FP_rate": "False-positive rate"
    }
)

fig.update_traces(
    texttemplate="%{text:.1%}",
    textposition="outside"
)

fig.update_yaxes(
    tickformat=".0%",
    range=[
        0,
        max(0.05, fp_rates["FP_rate"].max() * 1.25)
    ]
)

fig.update_layout(
    width=700,
    height=550,
    template="plotly_white"
)

fig.show()


# ------------------------------------------------------------
# 3. Systematic sampling function
# ------------------------------------------------------------

def sample_examples(
    df,
    view,
    outcome,
    n=12,
    seed=42
):
    subset = df[
        (df["View"] == view)
        & (df["Outcome"] == outcome)
    ].copy()

    if len(subset) <= n:
        return subset

    return subset.sample(
        n=n,
        random_state=seed
    )


# ------------------------------------------------------------
# 4. Highest-confidence FP examples
# ------------------------------------------------------------

def top_false_positives(df, view, n=12):
    return (
        df[
            (df["View"] == view)
            & (df["Outcome"] == "FP")
        ]
        .sort_values("score", ascending=False)
        .head(n)
    )


for view in ["AP", "PA"]:

    print("\n" + "=" * 70)
    print(view, "HIGHEST-CONFIDENCE FALSE POSITIVES")
    print("=" * 70)

    display(
        top_false_positives(d, view)[
            ["score", "Age_years", "Sex"]
        ]
    )


# ------------------------------------------------------------
# 5. Image retrieval
# ------------------------------------------------------------

def get_image(index):
    """
    Retrieve image from the feature space already created
    in xraymodel.ipynb.
    """
    img = XRAY_IMAGE_FEATURES[index]

    img = np.asarray(img).squeeze()

    # Normalize only for visualization
    lo, hi = np.percentile(img, [1, 99])

    if hi > lo:
        img = np.clip(
            (img - lo) / (hi - lo),
            0,
            1
        )

    return img


# ------------------------------------------------------------
# 6. Plotly image panel
# ------------------------------------------------------------

def plot_image_panel(
    df,
    view,
    outcome,
    n=12,
    seed=42,
    title=None
):

    examples = sample_examples(
        df,
        view=view,
        outcome=outcome,
        n=n,
        seed=seed
    )

    n_cols = 4
    n_rows = math.ceil(len(examples) / n_cols)

    if title is None:
        title = f"{PATHOLOGY}: {view} {outcome}"

    fig = make_image_montage(
        examples,
        n_rows=n_rows,
        n_cols=n_cols,
        title=title
    )

    fig.show()

    return examples


# ------------------------------------------------------------
# 7. Create Plotly image montage
# ------------------------------------------------------------

def make_image_montage(
    examples,
    n_rows,
    n_cols,
    title
):

    subplot_titles = [
        (
            f"{row['View']} · {row['Outcome']}<br>"
            f"Score: {row['score']:.2f} | "
            f"Age: {row['Age_years']} | "
            f"Sex: {row['Sex']}"
        )
        for _, row in examples.iterrows()
    ]

    fig = make_subplots(
        rows=n_rows,
        cols=n_cols,
        subplot_titles=subplot_titles,
        horizontal_spacing=0.05,
        vertical_spacing=0.15
    )

    for i, (_, row) in enumerate(examples.iterrows()):

        r = i // n_cols + 1
        c = i % n_cols + 1

        img = get_image(row.name)

        # Ensure a 2D grayscale image
        img = np.asarray(img, dtype=np.float32).squeeze()

        fig.add_trace(
            go.Heatmap(
                z=img,
                colorscale="Gray",
                zmin=0,
                zmax=1,
                showscale=False,
                hovertemplate="Pixel value: %{z:.2f}<extra></extra>"
            ),
            row=r,
            col=c
        )

        fig.update_xaxes(
            visible=False,
            fixedrange=True,
            row=r,
            col=c
        )

        fig.update_yaxes(
            visible=False,
            autorange="reversed",
            fixedrange=True,
            row=r,
            col=c
        )

    fig.update_layout(
        title=dict(
            text=title,
            x=0.5,
            xanchor="center"
        ),
        width=1200,
        height=350 * n_rows,
        template="plotly_white",
        margin=dict(
            l=20,
            r=20,
            t=100,
            b=30
        )
    )

    return fig


# ------------------------------------------------------------
# 8. Required Plotly import for subplots
# ------------------------------------------------------------

from plotly.subplots import make_subplots


# ------------------------------------------------------------
# 9. THE FOUR KEY PANELS
# ------------------------------------------------------------

AP_FP = plot_image_panel(
    d,
    view="AP",
    outcome="FP",
    n=12,
    seed=RANDOM_SEED,
    title=f"{PATHOLOGY} — AP False Positives"
)

AP_TN = plot_image_panel(
    d,
    view="AP",
    outcome="TN",
    n=12,
    seed=RANDOM_SEED,
    title=f"{PATHOLOGY} — AP True Negatives"
)

AP_TP = plot_image_panel(
    d,
    view="AP",
    outcome="TP",
    n=12,
    seed=RANDOM_SEED,
    title=f"{PATHOLOGY} — AP True Positives"
)


# PA images

PA_FP = plot_image_panel(
    d,
    view="PA",
    outcome="FP",
    n=12,
    seed=RANDOM_SEED,
    title=f"{PATHOLOGY} — PA False Positives"
)

PA_TN = plot_image_panel(
    d,
    view="PA",
    outcome="TN",
    n=12,
    seed=RANDOM_SEED,
    title=f"{PATHOLOGY} — PA True Negatives"
)

PA_TP = plot_image_panel(
    d,
    view="PA",
    outcome="TP",
    n=12,
    seed=RANDOM_SEED,
    title=f"{PATHOLOGY} — PA True Positives"
)

Number of valid AP/PA negative cases: 9367
View  Outcome
AP    FP         1880
      TN         1662
      TP          578
PA    FP         1218
      TN         3529
      TP          500
dtype: int64



AP HIGHEST-CONFIDENCE FALSE POSITIVES


,score,Age_years,Sex
4925,0.988640,74.0,Male
538,0.985916,83.0,Female
9075,0.985185,53.0,Female
6050,0.985074,70.0,Female
7701,0.984865,90.0,Female
7452,0.984825,73.0,Male
3024,0.983443,90.0,Female
7034,0.982335,89.0,Female
7575,0.981352,90.0,Female
7261,0.980708,84.0,Female



PA HIGHEST-CONFIDENCE FALSE POSITIVES


,score,Age_years,Sex
3276,0.995429,81.0,Female
7637,0.989735,64.0,Male
6740,0.988179,81.0,Female
129,0.986461,81.0,Male
6841,0.984728,60.0,Female
7514,0.981299,81.0,Female
2405,0.981277,76.0,Female
254,0.981209,48.0,Female
5149,0.981190,90.0,Female
3791,0.980950,77.0,Male


In [15]:

# ============================================================
# 7. MODEL SCORE DISTRIBUTIONS — PLOTLY
# ============================================================

import plotly.graph_objects as go

plot_df = d.copy()

plot_df["Group"] = (
    plot_df["View"]
    + " "
    + plot_df["Outcome"]
)

groups = [
    "AP TN",
    "AP FP",
    "PA TN",
    "PA FP"
]

fig = go.Figure()

# Add one box plot per group
for group in groups:

    scores = plot_df.loc[
        plot_df["Group"] == group,
        "score"
    ]

    fig.add_trace(
        go.Box(
            y=scores,
            name=group,
            boxpoints=False,  # Hide outliers
            hovertemplate=(
                "Group: " + group +
                "<br>Score: %{y:.3f}" +
                "<extra></extra>"
            )
        )
    )

# Add FP threshold
fig.add_hline(
    y=FP_THRESHOLD,
    line_dash="dash",
    line_color="red",
    annotation_text=f"FP threshold = {FP_THRESHOLD}",
    annotation_position="top left"
)

# Layout
fig.update_layout(
    title=dict(
        text=(
            "Cardiomegaly model confidence: "
            "AP/PA false positives vs true negatives"
        ),
        x=0.5
    ),
    xaxis_title="Group",
    yaxis_title="Model score",
    yaxis=dict(
        range=[0, 1]
    ),
    template="plotly_white",
    width=1000,
    height=600,
    showlegend=False
)

fig.show()
